# Structured Extraction from Messy Text

Real-world text is mostly unstructured fluff — log lines, prose,
markdown — with occasional **structured islands**: function calls,
JSON fragments, status codes, coordinates.  This notebook shows how
to use `from_regex` to define extraction patterns that skip the fluff
and pull structured data straight into Loom-typed fields.

Pipeline:

```
raw text  ──regex.search()──▶  matched substring  ──RegexSchema.parse()──▶  typed dict
                                                                              │
           LoomModel / LoomUnion  ◀──from_regex()──  pattern string          │
                   │                                                          │
           LoomCompiler.build_head()                                         │
                   │                                                          ▼
               LoomHead  ◀─────────────────────────────  logits / loss / decode
```

We'll work through three scenarios of increasing complexity:

1. **Function calls** embedded in log output
2. **JSON-like config** fragments in a YAML/text stream
3. **Multi-format union** — a single scanner that recognises function calls,
   HTTP status lines, and coordinate pairs, ignoring everything else

In [1]:
import re
from collections import OrderedDict

import torch
import torch.nn as nn

from loomlib import (
    LoomModel, LoomUnion, LoomCompiler, LoomHead,
    Categorical, BitInteger, Boolean, Scalar,
)
from loomlib.compat.regex import from_regex, RegexSchema

## Utility: scan for structured islands

`from_regex` uses `fullmatch` — it expects the *entire* string to match.
When the structured data is embedded in surrounding fluff, we need a thin
scanner that uses `re.search` / `re.finditer` to locate matches, then
delegates to `RegexSchema.parse` on the matched substring.

The helper below does exactly that.

In [2]:
def scan(schema: RegexSchema, text: str) -> list[dict]:
    """Find all occurrences of the pattern in *text* and parse each match."""
    results = []
    for m in schema.pattern.finditer(text):
        raw = m.group(0)
        # parse() expects a fullmatch, so we re-match the captured substring
        results.append(schema.parse(raw))
    return results


def scan_first(schema: RegexSchema, text: str) -> dict | None:
    """Return the first match, or None."""
    hits = scan(schema, text)
    return hits[0] if hits else None

---
## Scenario 1 — Function calls in log output

Imagine a log stream like:

```
[2024-01-15 08:32:01] INFO  Warming up caches...
[2024-01-15 08:32:04] DEBUG move(N, 42)
[2024-01-15 08:32:04] DEBUG cast(3, 100)
[2024-01-15 08:32:05] INFO  Flushing buffer
```

We only care about the `move(...)` and `cast(...)` calls.  Everything
else — timestamps, log levels, prose — is fluff to skip over.

In [3]:
fn_call = from_regex(
    r"move\((?P<direction>[NSEW]), (?P<speed>\d+)\)"
    r"|cast\((?P<spell_id>\d{1,3}), (?P<power>\d+)\)",
    name="FnCall",
)

print("Schema type:", "LoomUnion" if issubclass(fn_call.schema, LoomUnion) else "LoomModel")
print("Branches:  ", fn_call.schema.branch_names())
for bname, bmodel in fn_call.schema.loom_branches().items():
    print(f"  {bname}: {dict(bmodel.loom_fields())}")
print("Vocabularies:", fn_call.vocabularies)

Schema type: LoomUnion
Branches:   ['move', 'cast']
  move: {'direction': Categorical(4), 'speed': BitInteger(32)}
  cast: {'spell_id': BitInteger(10), 'power': BitInteger(32)}
Vocabularies: {'direction': ['N', 'S', 'E', 'W']}


In [4]:
log_text = """[2024-01-15 08:32:01] INFO  Warming up caches...
[2024-01-15 08:32:04] DEBUG move(N, 42)
[2024-01-15 08:32:04] DEBUG cast(3, 100)
[2024-01-15 08:32:05] INFO  Flushing buffer
[2024-01-15 08:32:06] DEBUG move(S, 7)
"""

for hit in scan(fn_call, log_text):
    print(hit)

{'direction': 0, 'speed': 42}
{'spell_id': 3, 'power': 100}
{'direction': 1, 'speed': 7}


Each parsed dict contains only the typed fields — the log timestamp,
level, and prose are never touched.  The `direction` field is a
Categorical index (0=N, 1=S, 2=E, 3=W); `speed`, `spell_id`, and
`power` are plain ints from BitInteger fields.

### Compile and decode

The extracted `LoomUnion` is a real Loom schema — we can compile it
into a head and run the full forward/decode pipeline.

In [5]:
head = LoomCompiler.build_head(fn_call.schema, d_model=64)
print(f"Total logits: {head.total_logits}")
print(f"Allocation:\n{head.allocation}")

# Simulate a forward pass
z = head(torch.randn(1, 64))
decoded = head.decode(z)
print("\nDecoded from random hidden state:")
for k, v in decoded.items():
    print(f"  {k}: {v}")

Total logits: 80
Allocation:
SliceAllocation(opcode=SliceEntry('__opcode__', z[0:2], Categorical(2)), branches={'move': BranchAllocation(name='move', entries=[SliceEntry('move.direction', z[2:6], Categorical(4)), SliceEntry('move.speed', z[6:38], BitInteger(32))]), 'cast': BranchAllocation(name='cast', entries=[SliceEntry('cast.spell_id', z[38:48], BitInteger(10)), SliceEntry('cast.power', z[48:80], BitInteger(32))])}, total_logits=80)

Decoded from random hidden state:
  __opcode__: tensor([1])
  move.direction: tensor([0])
  move.speed: tensor([148489712])
  cast.spell_id: tensor([716])
  cast.power: tensor([2450879488])


---
## Scenario 2 — JSON-like config fragments

Suppose a config stream has key-value pairs like
`"temperature": 0.85` scattered among comments and unstructured text.
We can extract specific numeric settings with a targeted regex.

In [6]:
kv_schema = from_regex(
    r'"(?P<key>temperature|top_p|max_tokens)": (?P<value>\d+\.\d+)',
    name="ConfigKV",
)

print("Fields:", dict(kv_schema.schema.loom_fields()))
print("Vocabs:", kv_schema.vocabularies)

Fields: {'key': Categorical(3), 'value': Scalar()}
Vocabs: {'key': ['temperature', 'top_p', 'max_tokens']}


In [7]:
config_blob = """
# Model configuration
{
    "model": "gpt-4",
    "temperature": 0.85,
    "top_p": 0.95,
    "max_tokens": 4096.0,
    "stream": true
}
"""

for hit in scan(kv_schema, config_blob):
    key_label = kv_schema.vocabularies["key"][hit["key"]]
    print(f"  {key_label} = {hit['value']}")

  temperature = 0.85
  top_p = 0.95
  max_tokens = 4096.0


The `"model"` and `"stream"` keys don't match the pattern — only the
three known numeric settings are extracted.  `key` is a `Categorical(3)`
with vocabulary `["temperature", "top_p", "max_tokens"]`; `value` is
a `Scalar()`.

---
## Scenario 3 — Multi-format union scanner

A single regex with top-level alternation can recognise *multiple*
structured formats simultaneously.  Each alternative becomes a branch
in a `LoomUnion` — the scanner skips everything that doesn't match
*any* branch.

In [8]:
multi = from_regex(
    # Branch 1: HTTP status line
    r"HTTP/(?P<http_ver>\d+\.\d+) (?P<status>\d{3})"
    r"|"
    # Branch 2: coordinate pair
    r"\((?P<lat>\d+\.\d+), (?P<lon>\d+\.\d+)\)"
    r"|"
    # Branch 3: function call
    r"(?P<fn_name>move|cast|heal)\((?P<arg>\d+)\)",
    name="MultiExtract",
)

print("Branches:", multi.schema.branch_names())
for bname, bmodel in multi.schema.loom_branches().items():
    print(f"  {bname}: {dict(bmodel.loom_fields())}")
print("Vocabularies:", multi.vocabularies)

Branches: ['http', 'branch_1', 'branch_2']
  http: {'http_ver': Scalar(), 'status': BitInteger(10)}
  branch_1: {'lat': Scalar(), 'lon': Scalar()}
  branch_2: {'fn_name': Categorical(3), 'arg': BitInteger(32)}
Vocabularies: {'fn_name': ['move', 'cast', 'heal']}


In [9]:
messy_text = """
Server responded HTTP/1.1 200 after 32ms.
User location: (37.7749, 122.4194) verified.
Executing move(42) on entity #7.
Some irrelevant log noise here...
Fallback to HTTP/2.0 503 — retrying.
heal(100) applied to player.
"""

for hit in scan(multi, messy_text):
    print(hit)

{'http_ver': 1.1, 'status': 200}
{'lat': 37.7749, 'lon': 122.4194}
{'fn_name': 0, 'arg': 42}
{'http_ver': 2.0, 'status': 503}
{'fn_name': 2, 'arg': 100}


Five structured islands extracted from six lines of mixed text.
The three different formats land in different union branches, each
with their own typed fields.

### Compile the union and inspect the allocation

In [10]:
multi_head = LoomCompiler.build_head(multi.schema, d_model=64)
print(f"Total logits: {multi_head.total_logits}")
print(f"\nAllocation:\n{multi_head.allocation}")

Total logits: 51

Allocation:
SliceAllocation(opcode=SliceEntry('__opcode__', z[0:3], Categorical(3)), branches={'http': BranchAllocation(name='http', entries=[SliceEntry('http.http_ver', z[3:4], Scalar()), SliceEntry('http.status', z[4:14], BitInteger(10))]), 'branch_1': BranchAllocation(name='branch_1', entries=[SliceEntry('branch_1.lat', z[14:15], Scalar()), SliceEntry('branch_1.lon', z[15:16], Scalar())]), 'branch_2': BranchAllocation(name='branch_2', entries=[SliceEntry('branch_2.fn_name', z[16:19], Categorical(3)), SliceEntry('branch_2.arg', z[19:51], BitInteger(32))])}, total_logits=51)


---
## Scenario 4 — Processing extracted data through the full pipeline

The real payoff: extracted dicts go straight into the encoder, through
a transformer, and out through the head — a complete structured
prediction loop where the regex handles the messy string boundary.

**Note:** `LoomEncoder.collate()` stores one scalar per token, so it
works with types whose `encode()` returns a single element: `Scalar`,
`ContinuousScalar`, `Boolean`.  We'll use a float+boolean pattern to
demonstrate the encoder path.

In [11]:
# A sensor schema with encoder-friendly types (Scalar, Boolean)
sensor = from_regex(
    r"temp=(?P<temperature>\d+\.\d+) ok=(?P<healthy>true|false)",
    name="SensorReading",
)

print("Fields:", dict(sensor.schema.loom_fields()))

encoder = LoomCompiler.build_encoder(sensor.schema, d_model=64)
head = LoomCompiler.build_head(sensor.schema, d_model=64)

print("Encoder branches:", encoder.branch_names)
print("Encoder fields:", {
    bname: list(fields.keys())
    for bname, fields in encoder.branch_fields.items()
})

Fields: {'temperature': Scalar(), 'healthy': Boolean()}
Encoder branches: ['__root__']
Encoder fields: {'__root__': ['temperature', 'healthy']}


In [12]:
# Extract sensor readings from a noisy log
sensor_log = """
[08:32:01] Calibrating...
[08:32:04] temp=23.5 ok=true
[08:32:05] temp=98.2 ok=false  << ALERT
[08:32:06] Recalibrating sensor array
[08:32:07] temp=24.1 ok=true
"""

raw_hits = scan(sensor, sensor_log)
print("Extracted readings:")
for h in raw_hits:
    print(f"  {h}")

# Flat model → each instance uses the "__root__" branch
instances = [("__root__", h) for h in raw_hits]
print("\nEncoder-ready instances:")
for inst in instances:
    print(f"  {inst}")

Extracted readings:
  {'temperature': 23.5, 'healthy': True}
  {'temperature': 98.2, 'healthy': False}
  {'temperature': 24.1, 'healthy': True}

Encoder-ready instances:
  ('__root__', {'temperature': 23.5, 'healthy': True})
  ('__root__', {'temperature': 98.2, 'healthy': False})
  ('__root__', {'temperature': 24.1, 'healthy': True})


In [13]:
# Collate → encode → transformer → head decode
batch = encoder.collate([instances])  # single-sequence batch
print(f"Batch shape: type_ids={batch.type_ids.shape}, values={batch.values.shape}")
print(f"Padding: {batch.padding_mask.sum().item()} padded tokens")

emb = encoder(batch)
print(f"Embedding shape: {emb.shape}")

# Tiny transformer
layer = nn.TransformerEncoderLayer(
    d_model=64, nhead=4, dim_feedforward=128, batch_first=True,
)
transformer = nn.TransformerEncoder(layer, num_layers=2)
hidden = transformer(emb, src_key_padding_mask=batch.padding_mask)

# Pool over sequence → single vector per batch element
real = ~batch.padding_mask
pooled = (hidden * real.unsqueeze(-1).float()).sum(dim=1) / real.sum(dim=1, keepdim=True).float()
z = head(pooled)
decoded = head.decode(z)

print("\nDecoded output (from untrained model):")
for k, v in decoded.items():
    print(f"  {k}: {v}")

Batch shape: type_ids=torch.Size([1, 6]), values=torch.Size([1, 6])
Padding: 0 padded tokens
Embedding shape: torch.Size([1, 6, 64])



Decoded output (from untrained model):
  temperature: tensor([-0.0395], grad_fn=<SqueezeBackward1>)
  healthy: tensor([False])


The full pipeline:

1. **Regex scan** — `scan()` finds structured islands in raw text
2. **`RegexSchema.parse()`** — converts matched substrings to typed dicts
3. **`LoomEncoder.collate()`** — packs dicts into columnar tensors
4. **`LoomEncoder.forward()`** — embeds into `[B, N, d_model]`
5. **Transformer** — processes the token sequence
6. **`LoomHead.decode()`** — maps logits back to typed values

The regex acts as the boundary between unstructured strings and Loom's
typed world.  Everything before the regex is free-form text;
everything after is well-typed tensors flowing through the model.

---
## Appendix: Inspecting type mappings

A quick reference for how regex sub-patterns map to Loom types.

In [14]:
examples = [
    (r"(?P<x>foo|bar|baz)",       "literal alternation"),
    (r"(?P<x>[NSEW])",            "character class"),
    (r"(?P<x>\d{3})",             "fixed-width digits"),
    (r"(?P<x>\d+)",               "unbounded digits"),
    (r"(?P<x>true|false)",         "boolean"),
    (r"(?P<x>\d+\.\d+)",          "float"),
]

print(f"{'Pattern':<30} {'Description':<25} {'LoomType':<20} {'Vocab'}")
print("─" * 95)
for pattern, desc in examples:
    rs = from_regex(pattern)
    lt = rs.schema.loom_fields()["x"]
    vocab = rs.vocabularies.get("x", "—")
    print(f"{pattern:<30} {desc:<25} {lt!r:<20} {vocab}")

Pattern                        Description               LoomType             Vocab
───────────────────────────────────────────────────────────────────────────────────────────────
(?P<x>foo|bar|baz)             literal alternation       Categorical(3)       ['foo', 'bar', 'baz']
(?P<x>[NSEW])                  character class           Categorical(4)       ['N', 'S', 'E', 'W']
(?P<x>\d{3})                   fixed-width digits        BitInteger(10)       —
(?P<x>\d+)                     unbounded digits          BitInteger(32)       —
(?P<x>true|false)              boolean                   Boolean()            —
(?P<x>\d+\.\d+)                float                     Scalar()             —
